In [ ]:
!pip install -q langchain langchain-openai langgraph-checkpoint-sqlite pandas

In [ ]:
import pandas as pd
import sqlite3

from IPython.display import Markdown
from google.colab import userdata
from langchain.agents import create_agent
from langchain.messages import HumanMessage
from langchain.tools import ToolRuntime, tool
from langchain_core.messages import BaseMessage
from langchain_core.runnables import RunnableLambda
from langchain_openai import ChatOpenAI
from langgraph.checkpoint.sqlite import SqliteSaver
from langgraph.store.sqlite import SqliteStore
from pydantic import SecretStr
from typing import List, TypedDict

api_key = SecretStr(userdata.get('OPENAI_API_KEY'))

def print_conversation(conversation: List[BaseMessage]):
    for message in conversation:
        message.pretty_print()

def explore_database(connection: sqlite3.Connection, table_name: str):
    result_df = pd.read_sql_query(f"SELECT * FROM \"{table_name}\"", connection)

    display(Markdown(f"## {table_name}"))
    display(result_df)

In [ ]:
class CustomAgentContext(TypedDict):
    user_id: str

In [ ]:
checkpointer_connection = sqlite3.connect("/content/checkpointer.db", check_same_thread=False, isolation_level=None)
checkpointer = SqliteSaver(checkpointer_connection)
checkpointer.setup()

store_connection = sqlite3.connect("/content/store.db", check_same_thread=False, isolation_level=None)
store = SqliteStore(store_connection)
store.setup()

In [ ]:
@tool
def remember_user_facts(key: str, value: str, runtime: ToolRuntime[CustomAgentContext]) -> str:
    """
    Extract durable user facts from a user message and store them in long-term memory. Example: "key: allergy; value: The user is allergic to nuts.", "key: hobbies; value: The user can play the guitar."

    Args:
        key: A unique identifier of the fact.
        value: The fact itself.
    """
    namespace = ("users", runtime.context["user_id"], "general_knowledge")
    prev_item = runtime.store.get(namespace, "auto_extracted_facts")

    facts_dict = prev_item.value if prev_item is not None else {}
    facts_dict[key] = value

    runtime.store.put(namespace, "auto_extracted_facts", facts_dict)
    return "OK"

@tool
def recall_user_facts(runtime: ToolRuntime[CustomAgentContext]) -> str:
    """
    Recall previously stored long-term facts about the user.
    """
    namespace = ("users", runtime.context["user_id"], "general_knowledge")
    result = runtime.store.search(namespace, limit=20)
    if not result:
        return "No facts stored."

    return '\n===\n'.join(f"{facts_group.key}:\n{'\n'.join(f'- {key}: \"{value}\"' for key, value in facts_group.value.items())}" for facts_group in result)

In [ ]:
agent = create_agent(
    model=ChatOpenAI(model="gpt-5-nano", api_key=api_key, reasoning_effort="low"),
    tools=[remember_user_facts, recall_user_facts],
    system_prompt=f"""
                  You are a polite and helpful personal assistant. You should use frequently the \"{remember_user_facts.name}\" tool to store information about the user that can be useful in future.
                  At the start of each iteration, ALWAYS use the \"{recall_user_facts.name}\" tool.
                  Be friendly - include known facts in the conversation to make the user feel special.
                  Proactively store useful data about the users - name, hobbies, plans, needs, etc.
                  """,
    checkpointer=checkpointer,
    store=store,
    context_schema=CustomAgentContext
)

interact = agent | RunnableLambda(lambda res: print_conversation(res["messages"]))

In [ ]:
interact.invoke(
    input={
        "messages": [HumanMessage("Hello! My name is Tony and I would like you to help me with the management of my personal notes and timeline.")]
    },
    config={
        "configurable": {
            "thread_id": "troeff_1"
        }
    },
    context={
        "user_id": "troeff"
    }
)

In [ ]:
explore_database(checkpointer_connection, "checkpoints")

In [ ]:
explore_database(store_connection, "store")

In [ ]:
# NOTE: We are passing a different `thread_id` - this is a different conversation (a few days after)
# The expected result - the agent should know (at least) my name.
interact.invoke(
    input={
        "messages": [HumanMessage("I want to plan a business meeting for today, 15:00.")]
    },
    config={
        "configurable": {
            "thread_id": "troeff_2"
        }
    },
    context={
        "user_id": "troeff"
    }
)

In [ ]:
interact.invoke(
    input={
        "messages": [HumanMessage("Hey! It's George. I need you to help me with the planning of a trip. I don't have much time left to waste.")]
    },
    config={
        "configurable": {
            "thread_id": "michael_3"
        }
    },
    context={
        "user_id": "michael"
    }
)

In [ ]:
explore_database(checkpointer_connection, "checkpoints")

In [ ]:
explore_database(store_connection, "store")